In [ ]:
import warnings
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd

project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [1]:
from pathlib import Path
import joblib

ROOT = Path("/Users/alexgonzalez/Documents/NBA-Prop-Predictor")
models = ROOT / "src/models/saved_models"
print("models dir:", models)
print("wnba files:", sorted(models.glob("*wnba*.joblib")))

min_b  = joblib.load(sorted(models.glob("min_nba_model_*.joblib"))[-1])
rate_b = joblib.load(sorted(models.glob("rpm_wnba_model_*.joblib"))[-1])
print("loaded", rate_b.keys())

models dir: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/models/saved_models
wnba files: [PosixPath('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/models/saved_models/apm_wnba_model_2026-07-12.joblib'), PosixPath('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/models/saved_models/ppm_wnba_model_2026-07-12.joblib'), PosixPath('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/models/saved_models/rpm_wnba_model_2026-07-16.joblib')]
loaded dict_keys(['quantile_models', 'feature_names', 'fold_metrics', 'holdout_metrics', 'train_end', 'val_end'])


In [8]:
from pathlib import Path
import joblib
from src.pipeline.predict import load_latest_odds, load_opp_def_ratings, predict_rate

league, date, prop = "wnba", "2026-07-17", "player_rebounds"
models = Path("src/models/saved_models")

min_b  = joblib.load(sorted(models.glob("min_nba_model_*.joblib"))[-1])
rate_b = joblib.load(sorted(models.glob("rpm_wnba_model_*.joblib"))[-1])

odds = load_latest_odds(league=league, region="dfs", prop=prop)
names = odds["NAME"].dropna().unique().tolist()
print("players:", names)

try:
    defs, avg_d, avg_p = load_opp_def_ratings(league=league)
except Exception as e:
    print("def ratings skipped:", e)
    defs, avg_d, avg_p = {}, None, None

preds = predict_rate(
    names, date, prop,
    league=league,
    min_bundle=min_b,
    rate_bundle=rate_b,
    def_ratings=defs,
    league_avg_def_rtg=avg_d,
    league_avg_pace=avg_p,
)
print(preds[["PLAYER_NAME", "OPP_TEAM", "HOME", "MIN_Q50", "RATE_Q50"]].to_string(index=False))

  raw.wnba_props_dfs: 609 rows  (pulled 2026-07-17 19:49:39)
players: ['Alyssa Thomas', 'Olivia Nelson-Ododa', 'DeWanna Bonner', 'Aaliyah Edwards', 'Kennedy Burke', 'Kahleah Copper', 'Jonquel Jones', 'Breanna Stewart', 'Sabrina Ionescu', 'Emily Engstler', 'Courtney Williams', 'Sarah Ashlee Barker', 'Olivia Miles', 'Kiki Iriafen', 'Shakira Austin', 'Kayla Thornton', 'Sonia Citron', 'Janelle Salaun', 'Brittney Griner']
  Loaded WNBA def ratings for 13 teams  |  avg DEF_RTG=103.1  |  avg PACE=94.8
  silver.wnba_player_gamelogs: 3,615 rows (season=2026)
        PLAYER_NAME OPP_TEAM  HOME  MIN_Q50  RATE_Q50
      Alyssa Thomas      CON     1    34.68    0.2037
Olivia Nelson-Ododa      PHX     0    27.37    0.2453
     DeWanna Bonner      CON     1    29.66    0.1804
    Aaliyah Edwards      PHX     0    16.63    0.2176
      Kennedy Burke      PHX     0    22.27    0.1422
     Kahleah Copper      CON     1    34.09    0.1136
      Jonquel Jones      IND     0    31.08    0.2713
    Breanna 

In [17]:
from src.pipeline.predict import load_latest_odds, line_probs_for_market
from src.utils.distributions import run_pts_simulation, run_count_simulation

# after you already have `preds` from predict_rate(...)
prop = "player_rebounds"  # or player_points / player_assists

lines = load_latest_odds(league="wnba", region="dfs", prop=prop)

sim_fn = {
    "player_points":   run_pts_simulation,
    "player_rebounds": run_count_simulation,
    "player_assists":  run_count_simulation,
}[prop]

# Per book: only score players that book actually prices (avoids LINE/P_* NaNs)
all_results = []
for book, book_lines in lines.groupby("BOOKMAKER"):
    book_lines = book_lines.dropna(subset=["NAME", "LINE"]).copy()
    # one LINE per player (Over/Under share the same number)
    book_lines = book_lines.drop_duplicates(subset=["NAME"], keep="first")

    priced = set(book_lines["NAME"].str.strip())
    preds_book = preds[preds["PLAYER_NAME"].isin(priced)]
    if preds_book.empty:
        continue

    results = line_probs_for_market(preds_book, book_lines, sim_fn, n_sims=10_000)
    results["BOOKMAKER"] = book
    all_results.append(results)

    print(book, f"({len(results)} priced)")
    display(results[["PLAYER_NAME", "LINE", "STAT_Q50", "P_OVER", "P_UNDER"]].head())

results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


  raw.wnba_props_dfs: 609 rows  (pulled 2026-07-17 19:49:39)
Betr DFS (15 priced)


,PLAYER_NAME,LINE,STAT_Q50,P_OVER,P_UNDER
0,Olivia Nelson-Ododa,5.5,6.71,0.623,0.377
1,Aaliyah Edwards,5.5,3.62,0.287,0.713
2,Kennedy Burke,3.5,3.17,0.423,0.577
3,Jonquel Jones,9.0,8.43,0.508,0.406
4,Breanna Stewart,7.5,7.57,0.524,0.476


DraftKings Pick6 (2 priced)


,PLAYER_NAME,LINE,STAT_Q50,P_OVER,P_UNDER
0,DeWanna Bonner,5.5,5.35,0.604,0.396
1,Brittney Griner,5.5,6.56,0.535,0.465


PrizePicks (18 priced)


,PLAYER_NAME,LINE,STAT_Q50,P_OVER,P_UNDER
0,Alyssa Thomas,7.5,7.06,0.450,0.550
1,Olivia Nelson-Ododa,6.5,6.71,0.523,0.477
2,DeWanna Bonner,5.5,5.35,0.584,0.416
3,Aaliyah Edwards,5.5,3.62,0.289,0.711
4,Kennedy Burke,3.5,3.17,0.415,0.585


Underdog (4 priced)


,PLAYER_NAME,LINE,STAT_Q50,P_OVER,P_UNDER
0,Breanna Stewart,7.5,7.57,0.528,0.471
1,Sabrina Ionescu,3.5,4.02,0.522,0.478
2,Kiki Iriafen,9.5,8.81,0.433,0.567
3,Janelle Salaun,3.5,4.00,0.546,0.454
